### BayFlux Demo Notebook: Infer fluxes from Fig. 3 of Antoniewicz 2007
By Tyler W. H. Backman

This notebook demonstrates Bayesian MCMC sampling from the simple test model in Fig. 3 of Antoniewicz 2007 (see citation below). We intend this notebook to serve as a template for use in more complicated projects.

In this notebook we import data from the following four files, created in an accompanying notebook:
* **File 1:** Metabolic model (.sbml format)
* **File 2:** Extracellular exchange and flux bounds (.csv format)
* **File 3:** Atom transitions for each reaction (.txt format)
* **File 4:** Mass distribution experimental data (.tab format)

Citation:

Antoniewicz MR, Kelleher JK, Stephanopoulos G. Elementary metabolite units (EMU): a novel framework for modeling isotopic distributions. Metab Eng. 2007;9(1):68-86. doi:10.1016/j.ymben.2006.09.001

#### Load Python libraries

In [ ]:
%matplotlib inline

In [ ]:
import os
import cobra
from cobra.sampling import MCMCACHRSampler
import bayflux
import math
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
print(np.__version__)
print(cobra.__version__)
print(bayflux.__version__)

#### Filenames for data import

In [ ]:
# Data directory
dataDir = 'input_data'
outputDataDir = os.path.join(dataDir, 'outputdata')
# Ensure both directories exist
os.makedirs(outputDataDir, exist_ok=True)

# File 1: Metabolic model (.sbml format)
modelFile = os.path.join(dataDir, 'fig3toyModel.xml')

# File 2: Extracellular exchange and flux bounds (.csv format)
fluxBoundsFile = os.path.join(dataDir, 'fig3toyFluxBounds.csv')

# File 3: Atom transitions for each reaction (.txt format)
transitionsFile = os.path.join(dataDir, 'fig3toyTransitions.txt')

# File 4: Mass distribution experimental data (.tab format)
mdvFile = os.path.join(dataDir, 'fig3toyMassDistribution.tab')

#### Import model and set flux bounds

In [ ]:
## Read the model file
cobrapymodel = cobra.io.read_sbml_model(modelFile)
model = bayflux.ReactionNetwork(cobrapymodel)
model

Import and apply exchange fluxes from file

In [ ]:
model.readFluxConstraints(fluxBoundsFile)

Set some flux bounds to limit the search space of the model. For test purposes we have narrow bounds here
so the notebook runs quickly, for real world models much larger bounds may be necessary and reasonable.

In [ ]:
# set maximum flux to 1.5x the input flux, to limit the search space
carbonInputReaction = model.reactions.get_by_id('a_exchange')
limit = 1.5 * -carbonInputReaction.lower_bound
for reaction in model.reactions:
    if reaction.lower_bound < -limit:
        reaction.lower_bound = -limit
    if reaction.upper_bound > limit:
        reaction.upper_bound = limit

Run FBA (Flux Balance Analysis) to confirm that the model is valid and can carry flux

In [ ]:
model.optimize()

#### Checking does all the reactions in the model are present in the atom transition file

In [ ]:
transitionsDF = pd.read_csv(transitionsFile)
transitionsDF

In [ ]:
# create set of all reaction IDs in iJN1463
modelReactionIds = {r.id for r in model.reactions}
len(modelReactionIds)

In [ ]:
# create set of all reaction IDs in atom transition file
reaction_names = transitionsDF.iloc[:, 0]
# Convert the extracted column to a set to get unique reaction names
transitionReactionIds = set(reaction_names)
# Display the set of reaction names
len(transitionReactionIds)

In [ ]:
# create set of all reactions in transition file without exact name matches to iJN1463
reactionIdDifference = transitionReactionIds.difference(modelReactionIds)
len(reactionIdDifference)

#### Read atom transitions and apply to model

In [ ]:
model.readAtomTransitions(transitionsFile)

#### Read mass distribution data

In [ ]:
importedMDVs = bayflux.readMassDistribution(model, mdvFile)
importedMDVs

#### Specify the number of extra neutrons in each position of the carbon inputs

In [ ]:
substrateLabelingDict = {
    model.metabolites.get_by_id('A'): ((1.0, [0, 1, 0]),),
}

#### Pre-compile EMU network

In [ ]:
compiledData = bayflux.emuCompile(list(importedMDVs.data.keys()), model, substrateLabelingDict)

#### Placeholder: These functions will be moved into the main package before final release

In [ ]:
def normpdf(x, mean, sd):
    # based on this but with np.log added https://codereview.stackexchange.com/a/98891
    # much faster, bur same answer as norm.logpdf
    var = float(sd)**2
    denom = (2*np.pi*var)**.5
    num = math.exp(-(float(x)-float(mean))**2/(2*var))
    return np.log(num/denom)

def logLikelihood(fluxVector, lcmsResults=importedMDVs, compiledData=compiledData, substrateLabelingDict=substrateLabelingDict, errorSD=0.010):

    # simulate labeling distribution
    try:
        results = bayflux.simulateLabeling(compiledData, fluxVector, substrateLabelingDict)
    except np.linalg.LinAlgError:
        print('LinAlgError')
        return np.finfo(np.float32).min # return a really low probability if we get an error
    
    logLikelihood = 0
    
    # find overlap between results and predicted EMUs
    for emu, mdvs in lcmsResults.data.items():
        emuHash = hash(emu)
        found = False
        for singleSize in compiledData:
            if emuHash in singleSize['matrixCoords']['internalEMUs']['hashes']:
                index = singleSize['matrixCoords']['internalEMUs']['hashes'].index(emuHash)
                found = True
                # zip together experimental results and simulated results into pairwise
                # tuples
                for mdv in mdvs:
                    for comparison in zip(mdv, results[singleSize['size']][index]):
                        logLikelihood += normpdf(comparison[0], comparison[1], errorSD)
        assert found == True, 'missing MDV in simulated results for ' + repr(emu)
    return logLikelihood

In [ ]:
def pairPlot(samples, reactions, thinning=1):
    # plot sample distribution for warmup samples
    g = sns.PairGrid(samples[reactions].iloc[::thinning, :], diag_sharey=False) # plot only every 10th sample
    # g = sns.PairGrid(samples[reactions], diag_sharey=False)
    g.map_lower(plt.hexbin, gridsize=25, cmap=plt.get_cmap('Blues'), linewidths=0)
    g.map_diag(sns.kdeplot, lw=3, gridsize=100)
    for i, j in zip(*np.triu_indices_from(g.axes, 1)):
        g.axes[i, j].set_visible(False)

#### Initialize sampler and collect uniform samples to identify the center of the metabolic model

In [ ]:
# initialize the sampler
mcmcachr = cobra.sampling.MCMCACHRSampler(model, thinning=1, seed=43)

In [ ]:
# collect uniform samples in order to identify the center of the model
centeringSamples = mcmcachr.sample(100000, fluxes=False, validatecheck=True)

In [ ]:
# show sample validation (1.0 = valid)
tol = 1e-6
pd.Series([int(x) for x in (mcmcachr.validate(centeringSamples, feas_tol=tol, bounds_tol=tol) == 'v')]).plot()

In [ ]:
# create a list of the reactions we want to plot fluxes for
reactionsToPlot = ['b_ec', 'd_f', 'b_d', model.reactions.get_by_id('b_d').reverse_variable.name]

In [ ]:
# inspect mixing in the last  warmup samples
centeringSamples[reactionsToPlot][-500:].plot()

In [ ]:
pairPlot(centeringSamples, reactionsToPlot)

#### Infer 13C MFA Posterior distribution from 13C Data using MCMC

In [ ]:
posteriorSamples = mcmcachr.sample(100000, fluxes=False, likelihood=logLikelihood, validatecheck=True)

In [ ]:
posteriorSamples

In [ ]:
# show frequency of validtions
# from cobrapy code:
#            - 'v' means feasible in bounds and equality constraints
#            - 'l' means a lower bound violation
#            - 'u' means a lower bound validation
#            - 'e' means and equality constraint violation
            
validationResult = pd.Series(mcmcachr.validate(centeringSamples))
validationResult.value_counts()

In [ ]:
# show sample validation (1.0 = valid)
tol = 1e-6
pd.Series([int(x) for x in (mcmcachr.validate(posteriorSamples, feas_tol=tol, bounds_tol=tol) == 'v')]).plot()

In [ ]:
# realSamples[allRxns].plot()
posteriorSamples[reactionsToPlot].plot()

In [ ]:
pairPlot(posteriorSamples, reactionsToPlot)

#### Compare best sample predicted labeling to published metabolite F MDV of [0.0001, 0.8008, 0.1983, 0.0009] and fluxes of {'a_b': [100, 0], 'b_d': [110, 50], 'b_ec': [20, 0], 'bc_de': [20, 0], 'd_f': [80, 0]}

In [ ]:
# Inspect the likeihood of the best sample collected
display(mcmcachr.bestSample[0])

In [ ]:
# Inspect the flux distributions of the best sample collected
pd.DataFrame(mcmcachr.bestSample[1], index=[v.name for v in model.variables], columns=['Flux'])

In [ ]:
# Inspect the predicted 13C labeling distributions for the 3 carbon metabolites
results = bayflux.simulateLabeling(compiledData, mcmcachr.bestSample[1], substrateLabelingDict)
pd.DataFrame(results[3], index=compiledData[2]['matrixCoords']['internalEMUs']['text'])

## Compute rhat

In [ ]:
### calculate 4 independent chains

warmupsamples = 100000
posteriorsamples = 100000
thinning = 1

In [ ]:
posteriorSampleArray = []

for chain in range(0,4):
    print(chain)
    
    # initialize the sampler
    seed = 43 + (chain * 100)
    print(f"seed: {seed}")
    mcmcachr = cobra.sampling.MCMCACHRSampler(model, thinning=thinning, seed=seed)
    
    # collect uniform samples in order to identify the center of the model
    centeringSamples = mcmcachr.sample(warmupsamples, fluxes=False, validatecheck=True)
    
    posteriorSamples = mcmcachr.sample(posteriorsamples, fluxes=False, likelihood=logLikelihood, validatecheck=True)
    posteriorSampleArray.append(posteriorSamples)
    
    # Save the posterior samples for this chain
    filename = os.path.join(outputDataDir, f'job{chain+1}_output.npy')
    np.save(filename, posteriorSamples, allow_pickle=False)
    print(f"Posterior samples for chain {chain} saved to {filename}")

In [ ]:
realSamples = np.concatenate(posteriorSampleArray)
realSamples.shape

In [ ]:
model.reactions

In [ ]:
# Tyler's code to get Christinas code working in this notebook
import timeit
import arviz as az
samplesPerTask = posteriorsamples

numpysamples = np.array(realSamples)
data = np.subtract(numpysamples[:,0::2], numpysamples[:,1::2])
posteriornetSamples = pd.DataFrame(data, columns=[v.id for v in model.reactions])
del data
posteriornetSamples1 = posteriornetSamples.loc[0:samplesPerTask-1,:]

# exclude these from consideration of rhat
excllist = ['a_exchange']

In [ ]:
# Filter fluxes to just the ones with variances


rxnVar=[]
#check which reactions have variance, we are just checking the first chain!
for i in model.reactions:
    if posteriornetSamples1.loc[:,i.id].std(axis=0)>0.001:
        if i.id not in excllist:
            rxnVar.append(i.id)   
display(posteriornetSamples)
posteriornetSamples_red = posteriornetSamples.loc[:,rxnVar]
del posteriornetSamples1
del posteriornetSamples

In [ ]:
display('The model has', len(rxnVar), 'rxns with Variance')

In [ ]:
posteriornetSamples_red1 = posteriornetSamples_red.iloc[0:samplesPerTask,:]
posteriornetSamples_red2 = posteriornetSamples_red.iloc[samplesPerTask:2*samplesPerTask,:]
posteriornetSamples_red3 = posteriornetSamples_red.iloc[2*samplesPerTask:3*samplesPerTask,:]
posteriornetSamples_red4 = posteriornetSamples_red.iloc[3*samplesPerTask:4*samplesPerTask,:]
posteriornetSamples_red1.shape

In [ ]:
#add burnin:
deletestart = int(samplesPerTask/5)
start_time1 = timeit.default_timer()
statsdata_rhat = np.zeros((len(rxnVar),1))
statsdata_ess = np.zeros((len(rxnVar),1))

#statsdata = pd.DataFrame(, columns=['mean', 'sd', 	hdi_3% 	hdi_97% 	mcse_mean 	mcse_sd 	ess_bulk 	ess_tail 	r_hat])
thin = 100
k = 1
j = 0
allinone = pd.DataFrame()
allinone_ess = pd.DataFrame()
break_out_flag = False

for sub in range(1, int((samplesPerTask-deletestart)/thin+1)):
    thisPoint = int(sub * thin + deletestart)
    if thisPoint>=deletestart:
        critnum = 0
        for i in range(0,len(rxnVar)):
            matrix_rhat = np.zeros((4,int(thisPoint-deletestart)))
            matrix_rhat = np.array((posteriornetSamples_red1.iloc[deletestart:thisPoint,i].to_numpy().transpose(), posteriornetSamples_red2.iloc[deletestart:thisPoint,i].to_numpy().transpose(), posteriornetSamples_red3.iloc[deletestart:thisPoint,i].to_numpy().transpose(), posteriornetSamples_red4.iloc[deletestart:thisPoint,i].to_numpy().transpose()))
            statsdata_rhat[i] = az.rhat(matrix_rhat)# default is "rank"#, method="split")#round_to=2)])
            
            #check whether 80% of fluxes have rhat<=1.1: if so, stop and return point, else stop
            if statsdata_rhat[i]<=1.1:
                critnum = critnum + 1
                if critnum >= .8*len(rxnVar):
                    print(thisPoint, 'samples after thinning are required for 80% of the reactions having an rhat of <=1.1.')
                    print(thisPoint*thinning, 'total samples (before thinning) are required for 80% of the reactions having an rhat of <=1.1.')
            
                    break_out_flag = True
                    break
            if break_out_flag:
                break
                
            statsdata_ess[i] = az.ess(matrix_rhat)
        sampnumb = (thisPoint-deletestart)*thinning
        allinone_ess = pd.concat([allinone_ess,pd.DataFrame(statsdata_ess, columns=[str(int(sampnumb))])],axis=1)
        allinone = pd.concat([allinone,pd.DataFrame(statsdata_rhat, columns=[str(int(sampnumb))])],axis=1)
        
elapsed1 = timeit.default_timer() - start_time1
print('comp time', elapsed1)


## Output analysis

In [ ]:
data = np.concatenate(posteriorSampleArray)
#warmupsamples = 100000
#posteriorsamples = 100000
# Print the shape of the combined data
print(f"Shape of combined data: {data.shape}")

# compute the shape of data
chains = int(data.shape[0] / posteriorsamples)
print('found ' + str(chains) + ' markov chains!')

In [ ]:
%%time

warmupSamples = mcmcachr.sample(warmupsamples, fluxes=False, likelihood=False)
warmupSamples.shape

In [ ]:
realSamples = pd.DataFrame(data, columns=[v.name for v in model.variables])
realSamples.shape

In [ ]:
realSampleThinning = math.ceil(realSamples.shape[0]/(posteriorsamples*chains))
realSampleThinning

In [ ]:
allRxns = [r.id for r in model.reactions][0:4]
allRxns

In [ ]:
realSamples[allRxns].iloc[0::realSampleThinning, :].plot()

In [ ]:
# plot sample distribution for samples
start = int((realSamples.shape[0]/10))
g = sns.PairGrid(realSamples[allRxns].iloc[start::realSampleThinning, :], diag_sharey=False) # plot only every 10th sample starting after sample 10k
# g = sns.PairGrid(realSamples[allRxns], diag_sharey=False)
g.map_lower(plt.hexbin, gridsize=25, cmap=plt.get_cmap('Blues'), linewidths=0)
g.map_diag(sns.kdeplot, lw=3, gridsize=100)
for i, j in zip(*np.triu_indices_from(g.axes, 1)):
    g.axes[i, j].set_visible(False)

### Plot marginal distributions for each reaction

Here we plot a marginal distribution for each reaction as a net flux, while excluding the first 10% of the samples, and by adding the forward and reverse fluxes to get net flux for reversible reactions.

In [ ]:
# summarize the samples to be plotted

deleteStart = 0
netSamples = np.subtract(realSamples.iloc[deleteStart::1,0::2], realSamples.iloc[deleteStart::1,1::2])
netUniform = np.subtract(warmupSamples.iloc[deleteStart::1,0::2], warmupSamples.iloc[deleteStart::1,1::2])
display(netUniform.shape)
display(netSamples.shape)

In [ ]:
# remove reactions with no variance, e.g. they cannot carry flux or were fully constrained

rxnsWithVariance = netUniform.std(axis=0) > 0.001
netUniform = netUniform.loc[:,rxnsWithVariance]
netSamples = netSamples.loc[:,rxnsWithVariance]

In [ ]:
display(netUniform.shape)
display(netSamples.shape)

In [ ]:
warmupSampleThinning = math.ceil(warmupSamples.shape[0]/warmupsamples)
warmupSampleThinning

In [ ]:
import matplotlib
# plot marginals

cols = 3
rows = math.ceil(netUniform.shape[1]/cols)

matplotlib.rcParams['figure.figsize'] = (16,3)
for j in range(0, rows):
    for k in range(0, cols):
        i = (j*cols) + k
        if i > (netUniform.shape[1] - 1):
            break
        # try:
        plt.subplot(1, cols, k + 1)
        sns.kdeplot(netUniform.iloc[0::warmupSampleThinning,i], lw=2, gridsize=100, color="grey", label="Uniform")
        sns.kdeplot(netSamples.iloc[0::realSampleThinning,i], lw=4, gridsize=100, color="darkred")
        # except:
            # continue

        plt.xlabel("Flux")
        if k == 0:
            plt.ylabel("Probability density")
    plt.show()

### Analyze sampler convergence with Gelman–Rubin diagnostic

Each chain sampling from the posterior effectively starts at a different random location from a uniform sample of the flux space because of the warmupSamples, which end at a different spot depending on the random seed. Values close to 1.0 (for example 1.1 or less) typically indicate convergence.

In [ ]:
samplesPerChain = posteriorsamples
samplesPerChain

In [ ]:
# compute stat with net fluxes only
# comment these out if we want to analyze each direction independently
numpysamples = np.array(realSamples)
data = np.subtract(numpysamples[:,0::2], numpysamples[:,1::2])
data.shape

In [ ]:
import arviz as az
import xarray as xr
#import concurrent.futures

# compute Gelman–Rubin diagnostic

# Define a number of 'burn in' samples which will be discarded, and not used to compute
# the GR diagnostic
burnin = int(samplesPerChain / 4)  # discard the first quarter samples

# Reshape data into a 3D array where each 1st dimension is a chain as used by pymc
samples = data.shape[0]
data3D = np.reshape(data, newshape=(chains, samplesPerChain, -1), order='C')
thin = int(math.ceil(samplesPerChain / 10))

# Initialize an array to store the Gelman-Rubin statistics
grStats = np.zeros(shape=(int((samplesPerChain - burnin) / thin), data3D.shape[2]), dtype=np.float64, order='C')

# Iteratively analyze convergence with stepsize thin using arviz's rhat function
for point in range(1, int((samplesPerChain - burnin) / thin) + 1):
    thisPoint = int(point * thin) + burnin
    subset = np.take(data3D, range(burnin + 1, thisPoint), axis=1)
    print(subset.shape)
    
    # Convert the subset into a DataArray and then an InferenceData object
    subset_xr = xr.DataArray(subset, dims=["chain", "draw", "variable"])
    idata = az.convert_to_inference_data(subset_xr)
    
    # Compute Gelman-Rubin diagnostic (R-hat) using arviz
    grStats[point - 1] = az.rhat(idata).to_array().values

In [ ]:
# plot the Gelman–Rubin diagnostic over the course of the simulation

matplotlib.rcParams['figure.figsize'] = (16,8)
# pd.DataFrame(grStats[100:,:]).plot(legend=False, ylim=(1,10))
pd.DataFrame(grStats).plot(legend=False, ylim=(1))

In [ ]:
# print out final convergence by reaction name
# we drop all NAs, as they are all reverse directions for non reversible reactions
GRbyVariable = pd.DataFrame(grStats[grStats.shape[0] - 1], index=[v.name for v in model.variables][0::2], columns=['GR']).dropna()
GRbyVariable[rxnsWithVariance].sort_values(by='GR')

In [ ]:
GRbyVariable.shape

In [ ]:
print(f"{(GRbyVariable['GR'] < 1.10).sum()} ({((GRbyVariable['GR'] < 1.10).sum()/GRbyVariable.shape[0])*100:.2f} %)")

In [ ]:
print(f"{(GRbyVariable['GR'] > 1.10).sum()} ({((GRbyVariable['GR'] > 1.10).sum()/GRbyVariable.shape[0])*100:.2f} %)")

In [ ]:
print(f"{(GRbyVariable[rxnsWithVariance] < 1.10).sum()}")

In [ ]:
print(f"{(GRbyVariable[rxnsWithVariance] > 1.10).sum()}")

## Analysis directly from saved files

In [ ]:
# Initialize an empty list to hold the posterior samples
posteriorSampleArray = []

# Load the posterior samples from the four different .npy files
for chain in range(4):
    filename = os.path.join(outputDataDir, f'job{chain}_output.npy')
    posteriorSamples = np.load(filename, allow_pickle=False)
    posteriorSampleArray.append(posteriorSamples)

# Concatenate the loaded posterior samples
data = np.concatenate(posteriorSampleArray)

# Print the shape of the combined data
print(f"Shape of combined data: {data.shape}")

# Compute the number of chains
chains = int(data.shape[0] / posteriorsamples)
print('found ' + str(chains) + ' markov chains!')

In [ ]:
%%time

warmupSamples = mcmcachr.sample(warmupsamples, fluxes=False, likelihood=False)
warmupSamples.shape

In [ ]:
realSamples = pd.DataFrame(data, columns=[v.name for v in model.variables])
realSamples.shape

In [ ]:
realSampleThinning = math.ceil(realSamples.shape[0]/(posteriorsamples*chains))
realSampleThinning

In [ ]:
allRxns = [r.id for r in model.reactions][0:4]
allRxns

In [ ]:
realSamples[allRxns].iloc[0::realSampleThinning, :].plot()

In [ ]:
# plot sample distribution for samples
start = int((realSamples.shape[0]/10))
g = sns.PairGrid(realSamples[allRxns].iloc[start::realSampleThinning, :], diag_sharey=False) # plot only every 10th sample starting after sample 10k
# g = sns.PairGrid(realSamples[allRxns], diag_sharey=False)
g.map_lower(plt.hexbin, gridsize=25, cmap=plt.get_cmap('Blues'), linewidths=0)
g.map_diag(sns.kdeplot, lw=3, gridsize=100)
for i, j in zip(*np.triu_indices_from(g.axes, 1)):
    g.axes[i, j].set_visible(False)

In [ ]:
# summarize the samples to be plotted

deleteStart = 0
netSamples = np.subtract(realSamples.iloc[deleteStart::1,0::2], realSamples.iloc[deleteStart::1,1::2])
netUniform = np.subtract(warmupSamples.iloc[deleteStart::1,0::2], warmupSamples.iloc[deleteStart::1,1::2])
display(netUniform.shape)
display(netSamples.shape)

In [ ]:
# remove reactions with no variance, e.g. they cannot carry flux or were fully constrained

rxnsWithVariance = netUniform.std(axis=0) > 0.001
netUniform = netUniform.loc[:,rxnsWithVariance]
netSamples = netSamples.loc[:,rxnsWithVariance]

In [ ]:
display(netUniform.shape)
display(netSamples.shape)

In [ ]:
warmupSampleThinning = math.ceil(warmupSamples.shape[0]/warmupsamples)
warmupSampleThinning

In [ ]:
import matplotlib
# plot marginals

cols = 3
rows = math.ceil(netUniform.shape[1]/cols)

matplotlib.rcParams['figure.figsize'] = (16,3)
for j in range(0, rows):
    for k in range(0, cols):
        i = (j*cols) + k
        if i > (netUniform.shape[1] - 1):
            break
        # try:
        plt.subplot(1, cols, k + 1)
        sns.kdeplot(netUniform.iloc[0::warmupSampleThinning,i], lw=2, gridsize=100, color="grey", label="Uniform")
        sns.kdeplot(netSamples.iloc[0::realSampleThinning,i], lw=4, gridsize=100, color="darkred")
        # except:
            # continue

        plt.xlabel("Flux")
        if k == 0:
            plt.ylabel("Probability density")
    plt.show()

In [ ]:
samplesPerChain = posteriorsamples
samplesPerChain

In [ ]:
# compute stat with net fluxes only
# comment these out if we want to analyze each direction independently
numpysamples = np.array(realSamples)
data = np.subtract(numpysamples[:,0::2], numpysamples[:,1::2])
data.shape

In [ ]:
import arviz as az
import xarray as xr
#import concurrent.futures

# compute Gelman–Rubin diagnostic

# Define a number of 'burn in' samples which will be discarded, and not used to compute
# the GR diagnostic
burnin = int(samplesPerChain / 4)  # discard the first quarter samples

# Reshape data into a 3D array where each 1st dimension is a chain as used by pymc
samples = data.shape[0]
data3D = np.reshape(data, newshape=(chains, samplesPerChain, -1), order='C')
thin = int(math.ceil(samplesPerChain / 10))

# Initialize an array to store the Gelman-Rubin statistics
grStats = np.zeros(shape=(int((samplesPerChain - burnin) / thin), data3D.shape[2]), dtype=np.float64, order='C')

# Iteratively analyze convergence with stepsize thin using arviz's rhat function
for point in range(1, int((samplesPerChain - burnin) / thin) + 1):
    thisPoint = int(point * thin) + burnin
    subset = np.take(data3D, range(burnin + 1, thisPoint), axis=1)
    print(subset.shape)
    
    # Convert the subset into a DataArray and then an InferenceData object
    subset_xr = xr.DataArray(subset, dims=["chain", "draw", "variable"])
    idata = az.convert_to_inference_data(subset_xr)
    
    # Compute Gelman-Rubin diagnostic (R-hat) using arviz
    grStats[point - 1] = az.rhat(idata).to_array().values

In [ ]:
# plot the Gelman–Rubin diagnostic over the course of the simulation

matplotlib.rcParams['figure.figsize'] = (16,8)
# pd.DataFrame(grStats[100:,:]).plot(legend=False, ylim=(1,10))
pd.DataFrame(grStats).plot(legend=False, ylim=(1))

In [ ]:
# print out final convergence by reaction name
# we drop all NAs, as they are all reverse directions for non reversible reactions
GRbyVariable = pd.DataFrame(grStats[grStats.shape[0] - 1], index=[v.name for v in model.variables][0::2], columns=['GR']).dropna()
GRbyVariable[rxnsWithVariance].sort_values(by='GR')